# Module 2 • Class 5 — Calculus, Probability, and Statistics  
## Level 1 Beginner — aligned with Dasha's final presentation

This notebook follows **Dasha's slide order**.

Every block is named by the slide it supports:

- **Slide 1-2:** setup and dataset
- **Inserted Colab block:** statistics basics that Dasha will not put on slides
- **Slides 3-8:** loss, MSE, slope, gradient descent, learning rate
- **Slide 9:** probability activity
- **Inserted Colab blocks:** distributions and bell curve
- **Slide 10:** conditional probability and Bayes
- **Slide 11:** hypothesis testing
- **Slides 12-13:** key takeaways and homework

### Main rule

Do not just run the cells.  
After every output, ask: **what does this number mean for a model?**


## Slide 1 — Title: Calculus, Probability, and Statistics

**What Dasha explains:**  
Today we connect three math ideas to ML:

- Calculus: how models update
- Probability: how models reason with uncertainty
- Statistics: how we understand data

**What we do in Colab:**  
Prepare the notebook.


In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["font.size"] = 12

print("Setup complete.")


## Slide 2 — Module 2 Overview

**What Dasha explains:**  
This is Class 5 inside Module 2: Programming and Mathematical Foundations.

**What we do in Colab:**  
Load the Superstore dataset, because we will use real columns like `Sales`, `Profit`, `Discount`, and `Quantity` for examples.

### Dataset loading — choose ONE source

Set `LOAD_MODE` in the next cell:

- `"preferred"` — if `superstore_cleaned.csv` or `SampleSuperstore.csv` is already uploaded
- `"manual_upload"` — upload CSV manually
- `"external_source"` — Kaggle / public fallback


In [ ]:
# @title Choose Data Source
LOAD_MODE = "preferred"  # @param ["preferred", "manual_upload", "external_source"]

PREFERRED_PATHS = [
    "superstore_cleaned.csv",
    "SampleSuperstore.csv",
    "Sample - Superstore.csv",
    "/content/superstore_cleaned.csv",
    "/content/SampleSuperstore.csv",
    "/content/Sample - Superstore.csv",
]

KAGGLE_DATASET = "vivek468/superstore-dataset-final"
PUBLIC_FALLBACK_URL = "https://raw.githubusercontent.com/dsrscientist/dataset1/master/superstore.csv"


def read_csv_safely(path_or_url):
    """Try common encodings used by Superstore CSV files."""
    last_error = None
    for enc in ["utf-8", "latin-1", "ISO-8859-1"]:
        try:
            return pd.read_csv(path_or_url, encoding=enc)
        except Exception as e:
            last_error = e
    raise last_error


def find_csv_file(folder):
    csvs = glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True)
    if not csvs:
        raise FileNotFoundError("No CSV file found in the downloaded dataset folder.")
    return csvs[0]


def load_superstore(load_mode=LOAD_MODE):
    if load_mode == "preferred":
        for path in PREFERRED_PATHS:
            if os.path.exists(path):
                print(f"Loading preferred local file: {path}")
                return read_csv_safely(path)
        raise FileNotFoundError(
            "Preferred file not found. Upload superstore_cleaned.csv / SampleSuperstore.csv, "
            "or set LOAD_MODE to 'manual_upload' or 'external_source'."
        )

    if load_mode == "manual_upload":
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise RuntimeError("No file uploaded.")
        filename = list(uploaded.keys())[0]
        print(f"Loading uploaded file: {filename}")
        return read_csv_safely(filename)

    if load_mode == "external_source":
        try:
            try:
                import kagglehub
            except ImportError:
                print("Installing kagglehub...")
                get_ipython().run_line_magic("pip", "-q install kagglehub")
                import kagglehub
            dataset_path = kagglehub.dataset_download(KAGGLE_DATASET)
            csv_path = find_csv_file(dataset_path)
            print(f"Loading Kaggle dataset file: {csv_path}")
            return read_csv_safely(csv_path)
        except Exception as e:
            print(f"Kaggle loading failed: {e}")
            print("Trying public CSV fallback...")
            return read_csv_safely(PUBLIC_FALLBACK_URL)

    raise ValueError("LOAD_MODE must be 'preferred', 'manual_upload', or 'external_source'.")


def normalize_column_name(name):
    return str(name).lower().replace(" ", "").replace("_", "").replace("-", "")


def find_column(df, possible_names):
    lookup = {normalize_column_name(c): c for c in df.columns}
    for name in possible_names:
        key = normalize_column_name(name)
        if key in lookup:
            return lookup[key]
    # fallback: substring match
    for c in df.columns:
        c_norm = normalize_column_name(c)
        if any(normalize_column_name(name) in c_norm for name in possible_names):
            return c
    raise KeyError(f"Could not find any of these columns: {possible_names}. Available columns: {list(df.columns)}")


df = load_superstore()

# Standard column references used below.
SALES_COL = find_column(df, ["Sales"])
PROFIT_COL = find_column(df, ["Profit"])
DISCOUNT_COL = find_column(df, ["Discount"])
QUANTITY_COL = find_column(df, ["Quantity"])

print(f"Loaded dataset: {df.shape[0]} rows, {df.shape[1]} columns")
print("Core columns:", SALES_COL, PROFIT_COL, DISCOUNT_COL, QUANTITY_COL)
df.head()


# Inserted Colab Block A — Statistics You Already Know  
## Goes after Slide 2, before Slide 3

**Dasha's note:** this block is not in the presentation, but it helps students before the model-learning part.

We quickly review the statistics students already saw or will need:

- **Mean:** average, sensitive to outliers
- **Median:** middle value, robust to outliers
- **Mode:** most frequent value
- **Standard deviation:** how spread out values are
- **Percentiles:** Q1, Q2, Q3

These are the numbers behind `df.describe()`.


In [ ]:
# Statistics You Already Know — with real Superstore Sales

sales = df[SALES_COL].dropna()

mean_sales = sales.mean()
median_sales = sales.median()
mode_sales = sales.mode().iloc[0]
std_sales = sales.std()
q1 = sales.quantile(0.25)
q2 = sales.quantile(0.50)
q3 = sales.quantile(0.75)

print("Sales statistics:")
print(f"Mean:   {mean_sales:,.2f}")
print(f"Median: {median_sales:,.2f}")
print(f"Mode:   {mode_sales:,.2f}")
print(f"Std:    {std_sales:,.2f}")
print(f"Q1:     {q1:,.2f}")
print(f"Q2:     {q2:,.2f}")
print(f"Q3:     {q3:,.2f}")

print("\nPandas describe():")
display(sales.describe())


In [ ]:
# Visual: histogram with mean and median

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(sales, bins=80, edgecolor="black", alpha=0.7)

ax.axvline(mean_sales, linestyle="--", linewidth=2, label=f"Mean = {mean_sales:,.0f}")
ax.axvline(median_sales, linestyle="--", linewidth=2, label=f"Median = {median_sales:,.0f}")

ax.set_title("Sales Distribution: Mean vs Median")
ax.set_xlabel("Sales")
ax.set_ylabel("Frequency")
ax.legend()
plt.show()


### Quick check — inserted stats block

Answer:

1. Which is larger for Sales: mean or median?
2. What does that tell us about the shape of the distribution?
3. Which value would you report as a “typical order” to a business stakeholder?


## Slide 3 — How does a model know how to improve?

**What Dasha explains:**  
A model improves only if it can measure how wrong it is.

**What we do in Colab:**  
Create simple actual and predicted values.

This is the first step toward understanding **loss**.


In [ ]:
# Slide 3: actual values vs model predictions

actual_sales = np.array([100, 200, 300])
predicted_sales = np.array([90, 250, 280])

comparison = pd.DataFrame({
    "Actual Sales": actual_sales,
    "Predicted Sales": predicted_sales,
})

display(comparison)


## Slide 4 — ML model's workflow

**What Dasha shows:**  
`DATA → MODEL → PREDICTION → ERROR → UPDATE → BETTER PREDICTION`

**What we do in Colab:**  
Turn this workflow into a tiny table.

**Key idea:**  
Before updating the model, we must calculate error.


In [ ]:
# Slide 4: from prediction to error

# Convention in this notebook:
# error = prediction - actual
# If the error is positive, the model predicted too high.
# If the error is negative, the model predicted too low.

errors = predicted_sales - actual_sales

workflow_table = pd.DataFrame({
    "Actual Sales": actual_sales,
    "Predicted Sales": predicted_sales,
    "Error = Prediction - Actual": errors,
})

display(workflow_table)

print("Mean error:", errors.mean())
print("But mean error can hide mistakes because positive and negative errors cancel out.")


## Slide 5 — The model can be wrong

**What Dasha explains:**  

Before improving a model, we need to measure how wrong it is.

- Low loss = better
- High loss = worse
- Loss = many model errors summarized into one number

**What we do in Colab:**  
Calculate errors, absolute errors, squared errors, and then MSE.

### Important note about sign

Sometimes people write:

- `error = prediction - actual`
- or `error = actual - prediction`

For MSE it does not matter, because we square the error.


In [ ]:
# Slide 5: errors and squared errors

errors = predicted_sales - actual_sales
absolute_errors = np.abs(errors)
squared_errors = errors ** 2

loss_table = pd.DataFrame({
    "Actual": actual_sales,
    "Predicted": predicted_sales,
    "Error": errors,
    "Absolute Error": absolute_errors,
    "Squared Error": squared_errors,
})

display(loss_table)

print("Mean absolute error:", absolute_errors.mean())
print("Mean squared error:", squared_errors.mean())


# Inserted Colab Block B — MSE  
## Goes immediately after Slide 5

**Dasha's note:** she will not put a separate MSE slide, but she will say it aloud.  
So we include it in Colab.

## What is MSE?

**MSE = Mean Squared Error**

It means:

1. calculate prediction error
2. square every error
3. take the average

In regression, especially **linear regression**, MSE is a common **loss function**.

Why ML models use it:

- it gives one number for model wrongness
- bigger mistakes become much more expensive
- the model tries to reduce this number during training


In [ ]:
# Inserted MSE block: calculate MSE manually

mse_manual = np.mean((predicted_sales - actual_sales) ** 2)

print("Actual sales:   ", actual_sales)
print("Predicted sales:", predicted_sales)
print("Errors:         ", predicted_sales - actual_sales)
print("Squared errors: ", (predicted_sales - actual_sales) ** 2)
print("\nMSE:", mse_manual)


In [ ]:
# Why MSE helps choose a better model

model_a_predictions = np.array([90, 250, 280])   # current model
model_b_predictions = np.array([105, 195, 310])  # another model

mse_a = np.mean((model_a_predictions - actual_sales) ** 2)
mse_b = np.mean((model_b_predictions - actual_sales) ** 2)

print("MSE Model A:", mse_a)
print("MSE Model B:", mse_b)

if mse_a < mse_b:
    print("Model A is better by MSE.")
else:
    print("Model B is better by MSE.")


### Quick check — MSE block

1. Which model had lower MSE?
2. Why does MSE punish large errors strongly?
3. Why is MSE useful in linear regression?

Expected idea:  
Linear regression can update its parameters to reduce MSE.


## Slide 6 — Slope = change

**What Dasha shows:**  

- Slope tells us how much `y` changes when `x` changes by 1.
- For a line, slope is constant.
- For a curve, the slope changes at different points.
- A derivative is the slope at one point of a curve.

**What we do in Colab:**  
Compare a line and a curve.


In [ ]:
# Slide 6 Part A: slope of a line

x1, y1 = 1, 3
x2, y2 = 4, 9

slope = (y2 - y1) / (x2 - x1)

print("Point A:", (x1, y1))
print("Point B:", (x2, y2))
print("Slope = (y2 - y1) / (x2 - x1)")
print("Slope:", slope)
print("\nMeaning: when x increases by 1, y increases by", slope)


In [ ]:
# Slide 6 Part B: line vs curve

x = np.linspace(-3, 3, 200)
line_y = 2 * x + 1
curve_y = x ** 2

fig, ax = plt.subplots(figsize=(9, 5))

ax.plot(x, line_y, label="Line: y = 2x + 1")
ax.plot(x, curve_y, label="Curve: y = x²")

ax.set_title("Line has constant slope; curve has changing slope")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.axhline(0, linewidth=1)
ax.axvline(0, linewidth=1)
ax.legend()
plt.show()


In [ ]:
# Slide 6 Part C: derivative intuition for y = x^2
# derivative of x^2 is 2x, so slope changes depending on x

points = np.array([-2, -1, 0, 1, 2])
slopes = 2 * points

pd.DataFrame({
    "x point": points,
    "slope of y = x² at this point": slopes
})


### Quick check for Slide 6

1. Is the slope of a line constant or changing?
2. Is the slope of a curve constant or changing?
3. Why do we need slope/derivative for model improvement?

Expected idea:  
The slope tells the model which direction to update.


## Slide 7 — Gradient Descent

**What Dasha explains:**  

Goal: find model parameters that minimize the loss.

Gradient descent:

1. start with random parameters
2. calculate the slope of the loss
3. take a small step downhill
4. repeat

**What we do in Colab:**  
Use a tiny fake loss curve to see how the parameter moves.


In [ ]:
# Slide 7: gradient descent on a simple loss curve
# We pretend the best parameter is w = 3.
# Loss is lowest when w is 3.

def loss(w):
    return (w - 3) ** 2

def gradient(w):
    return 2 * (w - 3)

w = -4.0          # starting parameter
learning_rate = 0.2
history = []

for step in range(12):
    current_loss = loss(w)
    current_gradient = gradient(w)
    history.append((step, w, current_loss, current_gradient))

    # update rule: move opposite to the slope
    w = w - learning_rate * current_gradient

history_df = pd.DataFrame(history, columns=["step", "w parameter", "loss", "gradient"])
display(history_df.round(4))


In [ ]:
# Visualize the path downhill

w_values = np.linspace(-5, 8, 300)
loss_values = loss(w_values)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(w_values, loss_values, label="Loss curve: (w - 3)²")

ax.scatter(history_df["w parameter"], history_df["loss"], s=60, label="Gradient descent steps")
for _, row in history_df.iterrows():
    ax.text(row["w parameter"], row["loss"] + 1, int(row["step"]), fontsize=9)

ax.set_title("Gradient Descent: moving toward lower loss")
ax.set_xlabel("parameter w")
ax.set_ylabel("loss")
ax.legend()
plt.show()


### Quick check for Slide 7

1. Where is the lowest point of this loss curve?
2. What happens to loss after each step?
3. What does the gradient tell us?

Expected idea:  
The gradient tells us the direction and steepness of the loss.


## Slide 8 — Learning Rate

**What Dasha shows:**  

Learning rate = step size.

- Too small: learning is very slow
- Good: learning moves toward the minimum
- Too large: learning can jump around or diverge

**What we do in Colab:**  
Try three learning rates on the same loss curve.


In [ ]:
# Slide 8: compare different learning rates

def run_gradient_descent(start_w, lr, steps=12):
    w = start_w
    rows = []
    for step in range(steps):
        rows.append({
            "step": step,
            "w": w,
            "loss": loss(w),
            "gradient": gradient(w),
            "learning_rate": lr,
        })
        w = w - lr * gradient(w)
    return pd.DataFrame(rows)

small_lr = run_gradient_descent(start_w=-4, lr=0.03, steps=12)
good_lr = run_gradient_descent(start_w=-4, lr=0.2, steps=12)
large_lr = run_gradient_descent(start_w=-4, lr=1.05, steps=12)

display(pd.concat([
    small_lr.assign(case="too small"),
    good_lr.assign(case="good"),
    large_lr.assign(case="too large")
]).head(12))


In [ ]:
# Visual: three learning-rate situations

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)

cases = [
    ("Too small", small_lr),
    ("Good", good_lr),
    ("Too large", large_lr),
]

for ax, (title, hist) in zip(axes, cases):
    ax.plot(w_values, loss_values)
    ax.scatter(hist["w"], hist["loss"], s=45)
    ax.set_title(title)
    ax.set_xlabel("w")
    ax.grid(True)

axes[0].set_ylabel("loss")
plt.suptitle("Learning rate controls the step size")
plt.tight_layout()
plt.show()


### Quick check for Slide 8

1. Which learning rate is too slow?
2. Which learning rate moves smoothly toward the minimum?
3. Which learning rate jumps too much?

Key phrase:  
Learning rate controls how big the update step is.


## Slide 9 — Probability

**What Dasha shows:**  
A section title: “Probability”.

**Dasha's note:**  
Probability basics should be an activity with questions.

**What we do in Colab:**  
Use questions first, then simulate a fair die.


# Probability Activity — before formulas

Discuss in pairs:

1. If we roll a fair die, what is the probability of getting 6?
2. What is the probability of getting an even number?
3. If 100 customers enter a store and 20 buy something, what is the probability of purchase?
4. Is probability always exact, or can we estimate it from data?

Write your answers before running the next code.


In [ ]:
# Slide 9: probability from counting outcomes

die_faces = np.array([1, 2, 3, 4, 5, 6])

prob_6 = 1 / len(die_faces)
prob_even = len([x for x in die_faces if x % 2 == 0]) / len(die_faces)

print("Probability of rolling a 6:", prob_6)
print("Probability of rolling an even number:", prob_even)

customers = 100
buyers = 20
purchase_probability = buyers / customers

print("\nProbability of purchase:", purchase_probability)


In [ ]:
# Simulation: roll a fair die many times

np.random.seed(42)

rolls = np.random.choice(die_faces, size=10_000)
estimated_prob_6 = np.mean(rolls == 6)
estimated_prob_even = np.mean(rolls % 2 == 0)

print("Estimated P(6):", estimated_prob_6)
print("Estimated P(even):", estimated_prob_even)

print("\nWith many trials, estimates get close to theoretical probabilities.")


# Inserted Colab Block C — Distributions: the shape of randomness  
## Goes after Slide 9, before Slide 10

**This block is from Darya's extra notes/screenshots, not from a visible slide.**

A distribution shows how values are spread.

Three common distributions:

- **Uniform:** every outcome is equally likely, like rolling a fair die
- **Normal:** bell curve, many values near the mean, like heights or test scores
- **Binomial:** number of successes in repeated trials, like churners out of 100 customers


In [ ]:
# Inserted distribution block: uniform, normal, binomial

np.random.seed(42)

uniform_data = np.random.randint(1, 7, size=5000)          # fair die rolls
normal_data = np.random.normal(loc=170, scale=10, size=5000)  # heights
binomial_data = np.random.binomial(n=100, p=0.2, size=5000)   # churners out of 100

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(uniform_data, bins=np.arange(1, 8) - 0.5, edgecolor="black")
axes[0].set_title("Uniform: fair die")
axes[0].set_xlabel("die outcome")

axes[1].hist(normal_data, bins=40, edgecolor="black")
axes[1].set_title("Normal: heights")
axes[1].set_xlabel("height")

axes[2].hist(binomial_data, bins=30, edgecolor="black")
axes[2].set_title("Binomial: churners out of 100")
axes[2].set_xlabel("number of churners")

plt.tight_layout()
plt.show()


### Distribution quick check

Which distribution fits each situation?

1. Rolling a fair die  
2. Heights of adults  
3. Number of churners out of 100 customers  

Expected answers:

1. Uniform  
2. Normal  
3. Binomial


# Inserted Colab Block D — The Bell Curve  
## Goes after the distributions block, before Slide 10

**This block is from Darya's extra notes/screenshots, not from a visible slide.**

The normal distribution is useful because many real-world measurements cluster around the mean.

Key ideas:

- mean = center
- standard deviation = spread
- 68-95-99.7 rule:
  - about 68% within 1 std
  - about 95% within 2 std
  - about 99.7% within 3 std


In [ ]:
# Bell curve example with mean and standard deviation

np.random.seed(42)

mu = 170
sigma = 10
heights = np.random.normal(mu, sigma, size=10_000)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(heights, bins=60, density=True, alpha=0.7, edgecolor="black")

for k in [1, 2, 3]:
    ax.axvline(mu - k*sigma, linestyle="--", linewidth=1)
    ax.axvline(mu + k*sigma, linestyle="--", linewidth=1)

ax.axvline(mu, linewidth=3, label="mean")

ax.set_title("Normal Distribution: Bell Curve")
ax.set_xlabel("height")
ax.set_ylabel("density")
ax.legend()
plt.show()

within_1 = np.mean((heights >= mu - sigma) & (heights <= mu + sigma))
within_2 = np.mean((heights >= mu - 2*sigma) & (heights <= mu + 2*sigma))
within_3 = np.mean((heights >= mu - 3*sigma) & (heights <= mu + 3*sigma))

print(f"Within 1 std: {within_1:.3f}")
print(f"Within 2 std: {within_2:.3f}")
print(f"Within 3 std: {within_3:.3f}")


### Bell curve quick check

1. What does the mean show?
2. What does standard deviation show?
3. If standard deviation is large, are values close together or spread out?


## Slide 10 — Conditional Probability and Bayes' Theorem

**What Dasha explains:**  

Conditional probability means:

> the probability of one event given that another event already happened.

Example:

A customer is more likely to leave if they had 3 complaints.

Bayes formula:

`P(A|B) = P(B|A) * P(A) / P(B)`

**What we do in Colab:**  
Use a small customer example.


In [ ]:
# Slide 10 Part A: conditional probability from counts

# Example customer groups:
# 1000 customers total
# 200 customers had 3+ complaints
# 120 of those customers churned

total_customers = 1000
customers_with_complaints = 200
churned_with_complaints = 120

p_churn_given_complaints = churned_with_complaints / customers_with_complaints

print("P(churn | 3+ complaints) =", p_churn_given_complaints)
print("Meaning: among customers with 3+ complaints, this fraction churned.")


In [ ]:
# Slide 10 Part B: Bayes theorem with a simple churn example

# A = customer will churn
# B = customer had 3+ complaints

P_A = 0.20       # prior: P(churn)
P_B_given_A = 0.60  # likelihood: P(complaints | churn)
P_B = 0.30       # evidence: P(complaints)

P_A_given_B = (P_B_given_A * P_A) / P_B

print("P(A) = P(churn):", P_A)
print("P(B|A) = P(complaints | churn):", P_B_given_A)
print("P(B) = P(complaints):", P_B)
print("\nP(A|B) = P(churn | complaints):", round(P_A_given_B, 3))


### Quick check for Slide 10

Match each term:

- `P(A)` = ?
- `P(B|A)` = ?
- `P(B)` = ?
- `P(A|B)` = ?

Expected idea:

- prior belief
- likelihood
- evidence
- updated belief / posterior


## Slide 11 — Hypothesis Testing

**What Dasha shows:**  

Model A accuracy = 84%  
Model B accuracy = 86%  

Question: which is better?

**Hypothesis testing asks:**  
Is the difference real, or could it be random noise?

- `H0` / null hypothesis: no real effect
- `H1` / alternative hypothesis: there is a real effect
- `p < 0.05` → reject H0


In [ ]:
# Slide 11: model comparison example

model_a_accuracy = 0.84
model_b_accuracy = 0.86

difference = model_b_accuracy - model_a_accuracy

print("Model A accuracy:", model_a_accuracy)
print("Model B accuracy:", model_b_accuracy)
print("Difference:", difference)

print("\nQuestion: is 2 percentage points a real improvement, or random noise?")


In [ ]:
# Beginner-friendly simulation:
# Suppose the true accuracy is actually 84%.
# How often could we see 86% or higher just by random chance?

np.random.seed(42)

n_customers = 500
true_accuracy_under_H0 = 0.84
observed_b_accuracy = 0.86

simulated_accuracies = np.random.binomial(
    n=n_customers,
    p=true_accuracy_under_H0,
    size=20_000
) / n_customers

p_value_simulated = np.mean(simulated_accuracies >= observed_b_accuracy)

print("Simulated p-value:", round(p_value_simulated, 4))

if p_value_simulated < 0.05:
    print("p < 0.05: reject H0. The improvement looks statistically meaningful.")
else:
    print("p >= 0.05: do not reject H0. The difference could be random noise.")


In [ ]:
# Visualize simulated accuracies under H0

fig, ax = plt.subplots(figsize=(10, 5))

ax.hist(simulated_accuracies, bins=40, edgecolor="black", alpha=0.7)
ax.axvline(observed_b_accuracy, linestyle="--", linewidth=3, label="Observed Model B accuracy")

ax.set_title("Could 86% happen by chance if true accuracy were 84%?")
ax.set_xlabel("simulated accuracy under H0")
ax.set_ylabel("frequency")
ax.legend()
plt.show()


### Quick check for Slide 11

1. What is H0 in this example?
2. What is H1?
3. If p-value is small, what do we do?
4. Does p-value prove that Model B is always better?

Expected idea:  
p-value helps us decide whether the observed difference is surprising under H0.


## Slide 12 — Key Takeaways

**What Dasha shows:**  

- Distributions — the shape of randomness
- The Bell Curve — why it is everywhere
- Bayes' Theorem — updating beliefs with evidence
- Next class: Class 6

**What we do in Colab:**  
Final recap.


In [ ]:
# Slide 12: final recap questions

questions = [
    "1. What does MSE measure?",
    "2. What does slope/derivative tell a model?",
    "3. What does learning rate control?",
    "4. What does a distribution show?",
    "5. What does P(A|B) mean?",
    "6. What is the null hypothesis?",
]

for q in questions:
    print(q)


### Write your final answers

1. MSE measures:  
2. A derivative tells the model:  
3. Learning rate controls:  
4. A distribution shows:  
5. `P(A|B)` means:  
6. `H0` means:


## Slide 13 — Thank you / Homework

**What Dasha shows:**  
Homework will be in the group.

**What we do in Colab:**  
Make sure students know what to review and what to submit.

### Beginner homework checklist

- [ ] I can explain error vs loss.
- [ ] I can calculate MSE from predictions and actual values.
- [ ] I can explain slope as change.
- [ ] I can explain gradient descent as moving downhill on loss.
- [ ] I can explain learning rate as step size.
- [ ] I can answer probability activity questions.
- [ ] I can explain conditional probability and Bayes terms.
- [ ] I can explain H0, H1, and p-value in simple words.


# Appendix — Original Class 5 notebook support: correlation

This section is not directly on Dasha's final slides, but it supports the original Class 5 homework theme: descriptive statistics and correlation.

Use it if there is time or for homework support.


In [ ]:
# Correlation matrix from original Level 1 notebook

core_numeric = df[[SALES_COL, PROFIT_COL, DISCOUNT_COL, QUANTITY_COL]].copy()
corr = core_numeric.corr()

corr.round(3)


In [ ]:
# Correlation heatmap

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True, ax=ax)
ax.set_title("Correlation Heatmap: Superstore Numeric Features")
plt.tight_layout()
plt.show()


### Correlation reflection

1. Which pair has the strongest correlation?
2. Is any relationship negative?
3. Does correlation prove causation?
4. Why can two highly correlated features be redundant in a model?
